In [41]:
df = pd.read_csv(r'C:\Users\12968\Downloads\plis_training.csv', sep='\t')

C:\Users\12968\AppData\Local\Temp\ipykernel_21844\2577409200.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\12968\Downloads\plis_training.csv', sep='\t')


### Column Names and Amount

In [42]:
import pandas as pd
df = pd.read_csv(r'C:\Users\12968\Downloads\plis_training.csv', sep='\t')
df['orderdate'] = pd.to_datetime(df['orderdate'])
print(df.shape, df.nunique())

C:\Users\12968\AppData\Local\Temp\ipykernel_21844\4013247051.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\12968\Downloads\plis_training.csv', sep='\t')


(8373695, 11) orderdate                        1096
legal_entity_id                 65511
set_id                        1388160
sku                           2617912
eclass                           8692
manufacturer                    20876
quantityvalue                    1076
vk_per_item                    336976
estimated_number_employees       2005
nace_code                         843
secondary_nace_code               708
dtype: int64


### Data range : 2023-01-01 to 2025-12-31, no out of range

In [43]:
import pandas as pd

df = pd.read_csv(r'C:\Users\12968\Downloads\plis_training.csv', sep='\t')

# Parse to datetime
df['orderdate'] = pd.to_datetime(df['orderdate'], errors='coerce')

# Check unparseable dates
print(f"Unparseable dates: {df['orderdate'].isna().sum()}")

# Check range
print(f"Min date: {df['orderdate'].min()}")
print(f"Max date: {df['orderdate'].max()}")

# Flag out-of-range (adjust bounds as needed)
cutoff_start = pd.Timestamp('2020-01-01')
cutoff_future = pd.Timestamp('2025-12-31')

out_of_range = df[(df['orderdate'] < cutoff_start) | (df['orderdate'] > cutoff_future)]
print(f"Out-of-range rows: {len(out_of_range)}")

C:\Users\12968\AppData\Local\Temp\ipykernel_21844\606249160.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\12968\Downloads\plis_training.csv', sep='\t')


Unparseable dates: 0
Min date: 2023-01-01 00:00:00
Max date: 2025-12-31 00:00:00
Out-of-range rows: 0


### Dupes

In [44]:
print(f"Total rows: {len(df)}")

Total rows: 8373695


In [45]:
# 1. Exact duplicate rows
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

# 2. Duplicate SKUs (same sku appearing multiple times)
sku_dupes = df.groupby('sku').size()
print(f"SKUs appearing more than once: {(sku_dupes > 1).sum()}")

# 3. Different SKUs mapping to same functional need (eclass + manufacturer)
functional = df.groupby(['eclass', 'manufacturer'])['sku'].nunique()
multi_sku = functional[functional > 1]
print(f"Eclass+Manufacturer combos with multiple SKUs: {len(multi_sku)}")
print(multi_sku.sort_values(ascending=False).head(10))

# 4. Same set_id or sku with conflicting eclass
conflicting = df.groupby('sku')['eclass'].nunique()
print(f"SKUs mapped to multiple eclass codes: {(conflicting > 1).sum()}")

Exact duplicate rows: 58173
SKUs appearing more than once: 1074202
Eclass+Manufacturer combos with multiple SKUs: 105088
eclass      manufacturer
23110102.0  Reyher          8570
21040301.0  Knipex          7505
24200101.0  HP              6886
21040401.0  Wera            6883
40011301.0  ELTEN           6841
23110102.0  Unbekannt       6741
24210590.0  Bedifol         6705
40011301.0  Atlas           6166
27050401.0  Varta           5996
24321202.0  Brother         5836
Name: sku, dtype: int64
SKUs mapped to multiple eclass codes: 122041


### dedup

In [46]:
df = df.drop_duplicates()
print(f"Rows after dedup: {len(df)}")

Rows after dedup: 8315522


### Missing value

In [47]:
# Missing values per column
print(df.isnull().sum())
print()
#print(f"% missing:\n{(df.isnull().mean() * 100).round(2)}")

orderdate                           0
legal_entity_id                     0
set_id                         575461
sku                                 0
eclass                          99933
manufacturer                    21195
quantityvalue                       0
vk_per_item                         0
estimated_number_employees     457611
nace_code                      130015
secondary_nace_code           4380894
dtype: int64



In [48]:
# Drop rows without eclass (critical for all prediction levels)
df = df.dropna(subset=['eclass'])
print(f"Rows after dropping missing eclass: {len(df)}")

# Fill missing manufacturer
df['manufacturer'] = df['manufacturer'].fillna('Unknown')

# Drop secondary_nace_code (>50% missing)
df = df.drop(columns=['secondary_nace_code'])

# Backfill buyer-level attributes from same legal_entity_id
for col in ['nace_code', 'estimated_number_employees']:
    df[col] = df.groupby('legal_entity_id')[col].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

print(f"Remaining missing:\n{df.isnull().sum()}")

Rows after dropping missing eclass: 8215589
Remaining missing:
orderdate                          0
legal_entity_id                    0
set_id                        548046
sku                                0
eclass                             0
manufacturer                       0
quantityvalue                      0
vk_per_item                        0
estimated_number_employees    451922
nace_code                     128477
dtype: int64


In [49]:
# How many buyers still have missing values
for col in ['nace_code', 'estimated_number_employees']:
    missing_buyers = df[df[col].isna()]['legal_entity_id'].nunique()
    total_buyers = df['legal_entity_id'].nunique()
    print(f"{col}: {missing_buyers}/{total_buyers} buyers fully missing")

nace_code: 12165/64898 buyers fully missing
estimated_number_employees: 30430/64898 buyers fully missing


### Manufacturers 

In [50]:
# Overview
print(f"Unique manufacturers: {df['manufacturer'].nunique()}")
print(f"Top 20:\n{df['manufacturer'].value_counts().head(20)}")

# Lowercase + strip whitespace
df['manufacturer'] = df['manufacturer'].str.strip().str.lower()

# Check "unbekannt" (unknown) variants
unknown_variants = df[df['manufacturer'].str.contains('unbek|unknown|unbekannt|n/a|n.a|keine|k.a', case=False, na=False)]['manufacturer'].value_counts()
print(f"\nUnknown-like manufacturers:\n{unknown_variants}")

# Standardize all unknown variants to single label
unknown_pattern = r'^(unbekannt|unknown|n/a|n\.a\.|keine|k\.a\.|nicht bekannt|no name|noname)$'
df['manufacturer'] = df['manufacturer'].replace(unknown_pattern, 'unknown', regex=True)

# Check after normalization
print(f"\nUnique after normalization: {df['manufacturer'].nunique()}")

Unique manufacturers: 20314
Top 20:
manufacturer
Unbekannt     292677
HP            121583
Soennecken    119269
Logitech       89679
Leitz          88838
Bosch          84706
3M             80747
tesa           78158
Brother        75248
Wera           75182
edding         71909
Varta          65097
Knipex         62467
LANDEFELD      60977
Henkel         58883
Durable        58482
Siemens        57511
Reyher         55511
Apple          49163
Riegler        47990
Name: count, dtype: int64

Unknown-like manufacturers:
manufacturer
unbekannt                                                                                                                 292677
phoenix contact                                                                                                            40106
wago kontakttechnik                                                                                                        25808
klauke                                                                      

In [52]:
# Check for potential duplicates with common prefixes
from collections import defaultdict

mfrs = df['manufacturer'].unique()

# Group by first 4 chars to find similar names
prefix_groups = defaultdict(list)
for m in mfrs:
    if m != 'unknown' and len(m) >= 4:
        prefix_groups[m[:4]].append(m)

# Show groups with multiple entries
fuzzy = {k: v for k, v in prefix_groups.items() if len(v) > 1}
print(f"Potential fuzzy duplicate groups: {len(fuzzy)}")

# Show top examples sorted by group size
for k, v in sorted(fuzzy.items(), key=lambda x: -len(x[1]))[:15]:
    print(f"\n'{k}': {v[:10]}")

Potential fuzzy duplicate groups: 3193

'ede ': ['ede - haromac', 'ede - riethmüller', 'ede - kleinbongartz', 'ede - 3m', 'ede - stanley', 'ede - wera', 'ede - sfs group', 'ede - fischer', 'ede - martor', 'ede - prosperplast']

'inte': ['intenso', 'intel', 'intellinet', 'intersnack', 'inter-tech', 'interflux', 'inter bär', 'interstuhl büromöbel', 'intersteel', 'intercable tools']

'euro': ['eurochron', 'europel', 'eurolub', 'eurotime uhrenvertrieb', 'eurobins', 'eurokraft', 'euro kuvert', 'europa-lehrmittel', 'eurolite', 'eurotronic']

'hans': ['hansa by styro', 'hansgrohe', 'hansi-siebert', 'hansa world of office', 'hansa', 'hans walz', 'hansaplast', 'hans turck', 'hans sasserath', 'hans wilms']

'ther': ['thermal grizzly', 'thermo fisher scientific', 'thermo electron led', 'thermo elect.led gmbh (samco)', 'thermohauser', 'thermorex', 'thermaflex', 'thermos l.l.c.', 'thermo future box', 'thermos']

'tech': ['technaxx', 'technotrade', 'techly', 'technotrenn', 'techtronic industries', '